**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Dimensionality Reduction — Simplifying Without Losing Signal

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
digits = load_digits()
X, y = digits.data, digits.target  # X shape: (1797, 64)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

## Step 1: Scale, Reduce, then Classify

A standard pipeline for high-dimensional data is: scale features → reduce with PCA → train a classifier. The code below runs this pipeline and reports test accuracy.

In [ ]:
# AI-generated — contains Bug 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # <- Bug 1
pca = PCA(n_components=30)
X_pca = pca.fit_transform(X_scaled)  # <- Bug 1 (also fit on full data)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, clf.predict(X_test)):.3f}")

**Bug 1 Investigation:** Run the cell above. What looks wrong about when the scaler and PCA are fit relative to the train/test split?

In [ ]:
# Fix Bug 1 here
# Split first, then fit_transform on X_train only, transform X_test
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: Choosing the Right Number of Components

After the correct split, the analyst picks a fixed number of PCA components and checks how much variance is explained.

In [ ]:
# AI-generated — contains Bug 2
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
X_train_sc = scaler2.fit_transform(X_train_raw)
X_test_sc = scaler2.transform(X_test_raw)

pca2 = PCA(n_components=30)  # <- Bug 2
pca2.fit(X_train_sc)
cum_var = np.cumsum(pca2.explained_variance_ratio_)
print(f"Variance explained by 30 components: {cum_var[-1]:.3f}")
print("Using 30 components as specified")

**Bug 2 Investigation:** Run the cell above. Does 30 components capture enough variance? How would you choose the number of components in a principled way?

In [ ]:
# Fix Bug 2 here
# Use PCA(n_components=0.95) to automatically retain 95% of variance
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Visualising Clusters with t-SNE

The analyst applies t-SNE to visualise how digit classes cluster in 2D. The code feeds data directly into t-SNE.

In [ ]:
# AI-generated — contains Bug 3
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_train_sc)  # <- Bug 3: raw 64-dim input to t-SNE
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train, cmap='tab10', alpha=0.6)
plt.colorbar(scatter, label='Digit class')
plt.title('t-SNE visualisation')
plt.show()

**Bug 3 Investigation:** Run the cell above (it may be slow). Why is passing 64-dimensional data directly into t-SNE problematic? What is the recommended preprocessing step before t-SNE?

In [ ]:
# Fix Bug 3 here
# First reduce to ~50 components with PCA, then pass that to TSNE
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below shows all three fixes applied together: proper train/test split before any preprocessing, variance-driven component selection, and PCA pre-reduction before t-SNE.

In [ ]:
# --- Fix 1: split first, then fit preprocessing only on training data ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_c = StandardScaler()
X_train_sc_c = scaler_c.fit_transform(X_train_r)
X_test_sc_c = scaler_c.transform(X_test_r)

# --- Fix 2: use n_components=0.95 to retain 95% variance ---
pca_c = PCA(n_components=0.95)
X_train_pca_c = pca_c.fit_transform(X_train_sc_c)
X_test_pca_c = pca_c.transform(X_test_sc_c)
print(f"Components to capture 95% variance: {pca_c.n_components_}")

clf_c = LogisticRegression(max_iter=1000, random_state=42)
clf_c.fit(X_train_pca_c, y_train_r)
print(f"Corrected accuracy: {accuracy_score(y_test_r, clf_c.predict(X_test_pca_c)):.3f}")

# --- Fix 3: PCA to 50 dims before t-SNE ---
pca_50 = PCA(n_components=50)
X_train_50 = pca_50.fit_transform(X_train_sc_c)

tsne_c = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne_c = tsne_c.fit_transform(X_train_50)

plt.figure(figsize=(8, 6))
scatter_c = plt.scatter(X_tsne_c[:, 0], X_tsne_c[:, 1], c=y_train_r, cmap='tab10', alpha=0.6)
plt.colorbar(scatter_c, label='Digit class')
plt.title('t-SNE visualisation (PCA pre-reduced to 50 dims)')
plt.show()